In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()
print(dataset)
with open('dataset-model.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [11]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [12]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [13]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    return {
        "output": output,
        "test_case": test_case,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "reasoning": reasoning,
        "score": score,
    }

In [14]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [ ]:
with open("dataset-model.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 6.666666666666667
[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): An S3 URI in the format 's3://bucket-name/key/path'\n        \n    Returns:\n        str: The bucket name\n        \n    Raises:\n        ValueError: If the URI is not in a valid S3 format\n    \"\"\"\n    if not s3_uri:\n        raise ValueError(\"S3 URI cannot be empty\")\n    \n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"S3 URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Find the bucket name (everything before the first '/')\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(\"S3 URI must contain a bucket name\")\n    \n 